# Role Drift Environment - GRPO Training

This notebook demonstrates training a small model (Qwen2.5-1.5B-Instruct) to resist role drift using GRPO on the Role Drift Environment.

## Setup

In [ ]:
!pip install -q openenv-core trl transformers accelerate datasets sentence-transformers fastapi uvicorn pydantic pytest
!git clone https://github.com/yourusername/role-drift-env.git
%cd role-drift-env
!pip install -e .

## Generate SFT Warm-Start Data

In [ ]:
from training.generate_sft_data import generate_sft_conversations

generate_sft_conversations(n_conversations=400)

## SFT Training

In [ ]:
from training.train_sft import train_sft

train_sft(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    data_path="data/sft/warmstart_conversations.jsonl",
    output_dir="checkpoints/sft",
    num_train_epochs=1.0,
    learning_rate=2e-5,
)

## GRPO Training

In [ ]:
from training.train_grpo import GRPOTrainer
import json

# Load scenario IDs
scenario_ids = []
with open("data/scenarios/train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        scenario_ids.append(obj["scenario_id"])

trainer = GRPOTrainer(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    sft_checkpoint="checkpoints/sft",
    output_dir="checkpoints/grpo",
    group_size=4,
    kl_coef=0.05,
)

trainer.train(scenario_ids, num_episodes=200)

## Evaluation

In [ ]:
from training.eval import evaluate_model

results = evaluate_model(
    model_path="checkpoints/grpo/best",
    scenario_file="data/scenarios/eval.jsonl",
    num_seeds=3,
    output_path="eval_results.jsonl",
)

## Plotting

In [ ]:
from scripts.plot_results import plot_reward_curve, plot_eval_comparison

plot_reward_curve()
plot_eval_comparison({
    "SFT-only": "eval_sft.jsonl",
    "GRPO": "eval_results.jsonl",
})